# 05 · Spatiotemporal Linking
Integrates Live-1, Live-2, intermediate fixed frame (Cy1), and multiplexed fixed data
into a single unified array. Computes alignment shifts two ways:
- **Live transitions**: manual fiducial annotation in napari
- **Fixed cycles**: Cellpose-guided DAPI phase cross-correlation
Shifts are applied to images, segmentation labels, and track coordinates.

> **Known issue**: a single large jump ~3 frames into the fixed alignment needs resolving.

In [ ]:
from pathlib import Path
from collections import defaultdict
from natsort import natsorted
import zarr
import dask.array as da
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.ndimage import shift as ndi_shift
from skimage.registration import phase_cross_correlation
from cellpose import models
import napari

## 1 · Initialise viewer and GPU model

In [ ]:
viewer = napari.Viewer(title='Linking')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cellpose_model = models.CellposeModel(gpu=(device == 'cuda'))

## 2 · Load Live-1

In [ ]:
def load_zarr_pos(zarr_dir):
    """Load images, segmentation, and track table for a given position zarr."""
    zarr_dir = Path(zarr_dir)
    root = zarr.open_group(zarr_dir, mode='r')
    images = da.asarray(root['images']['0'].images)
    seg    = da.asarray(root['labels']['trackastra_labels']['0'])
    tracks_path = zarr_dir / 'labels/trackastra_labels/tables/track'
    cols = [f.name for f in tracks_path.iterdir() if f.is_dir()]
    tracks = pd.DataFrame({col: zarr.open(tracks_path / col, mode='r')[:] for col in cols})
    return images, seg, tracks

live_1_images, live_1_segmentation, live_1_tracks = load_zarr_pos(
    '/mnt/DATA3/BPP0050/live_1.zarr/3_4'
)
print('Live-1 images:', live_1_images)
print('Live-1 segmentation:', live_1_segmentation)

## 3 · Load Live-2

In [ ]:
live_2_images, live_2_segmentation, live_2_tracks = load_zarr_pos(
    '/mnt/DATA3/BPP0050/live_2.zarr/3_4'
)
print('Live-2 images:', live_2_images)
print('Live-2 segmentation:', live_2_segmentation)

## 4 · Load multiplexed fixed images

In [ ]:
zarr_dir = Path('/mnt/DATA3/BPP0050/multiplexed.zarr/3_4')
zg = zarr.open_group(zarr_dir)
multiplex_images = da.asarray(zg.images)[:, 0, ...]  # collapse pseudo-timepoint axis
print('Multiplex images:', multiplex_images)

## 5 · Load intermediate fixed frame (Cy1)

In [ ]:
cy1_zarr = zarr.open_group(
    '/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Fixed_Cy1__2025-04-14T16_57_11-Measurement 1/acquisition/zarr/(3, 4).zarr',
    mode='r'
)
intermediate_frame = da.asarray(cy1_zarr['images'])
# Drop two unnecessary channels (keep DPC, BF, Mtb, DAPI)
intermediate_frame = intermediate_frame[:, [0, 1, 2, 5], :, :, :]
print('Intermediate frame:', intermediate_frame)

## 6 · Concatenate and flatten Z

In [ ]:
# Concatenate live acquisitions along time
live_images = da.concatenate([live_1_images, live_2_images], axis=0)

# Z-max projection
live_images        = live_images.max(axis=2)
intermediate_frame = intermediate_frame.max(axis=2)
multiplex_images   = multiplex_images.max(axis=2)

print('Live:', live_images.shape, '  Fixed:', intermediate_frame.shape, '  Multiplex:', multiplex_images.shape)

## 7 · Pad live channels to match fixed channel count
Live data has 3 channels; fixed has 4. A blank channel is appended to the live array.

In [ ]:
blank = da.zeros((live_images.shape[0], 1, *live_images.shape[-2:]), dtype=live_images.dtype)
live_images = da.concatenate([live_images, blank], axis=1)

images = da.concatenate([live_images, intermediate_frame, multiplex_images])
print('Unified array:', images)

## 8 · Concatenate segmentation
Live segmentation is extended across the fixed frames by repeating the final frame.

In [ ]:
live_segmentation = da.concatenate([live_1_segmentation, live_2_segmentation])
n_fixed = images.shape[0] - live_segmentation.shape[0]

segmentation = da.concatenate([
    live_segmentation,
    da.repeat(live_segmentation[-1:], n_fixed, axis=0)  # propagate last cytoplasmic mask
])
print('Segmentation:', segmentation)

## 9 · Load into memory

In [ ]:
images      = images.rechunk((10, 4, 1024, 1024)).compute()
segmentation = segmentation.rechunk((1, -1, -1)).compute()
print('Images:', images.shape, '  Segmentation:', segmentation.shape)

viewer.add_image(images, channel_axis=1, name='All frames')
viewer.add_labels(segmentation, name='Segmentation')

## 10 · Automatically compute fixed-cycle shifts
Cellpose segments the DAPI channel (frames 192–209); centroids guide phase cross-correlation.

In [ ]:
dapi_ch = -1
dapi_stack = images[192:210, dapi_ch, 2943:3967, 2943:3967]  # 1024x1024 centre crop

alignment_segmentations = []
fixed_shifts = np.zeros((210, 2), dtype=np.float32)

for i, img in enumerate(tqdm(dapi_stack, desc='Auto-aligning fixed frames')):
    masks, _, _ = cellpose_model.eval(img)
    alignment_segmentations.append(masks)
    if i > 0:
        shift, _, _ = phase_cross_correlation(
            alignment_segmentations[0].astype(float),
            masks.astype(float), upsample_factor=10
        )
        fixed_shifts[192 + i] = shift

dapi_seg = np.stack(alignment_segmentations)
v = napari.Viewer(title='Auto-alignment QC')
v.add_image(dapi_stack, name='DAPI stack')
v.add_labels(dapi_seg, name='DAPI segmentation')

## 11 · Manually annotate live-to-live shifts via napari fiducials
Add a 3D Points layer named `'fiducials'` in the viewer, place pairs of reference
points at the same cell across the transition frame (t=47 -> t=48, t=191 -> t=192),
then run the cell below.

In [ ]:
# Paste manually annotated fiducials here after napari annotation:
fiducials = np.asarray([
    [ 47., 3222.14, 3368.90],
    [ 48., 2994.20, 2616.17],
    [ 47., 3065.77, 3392.76],
    [ 48., 2832.53, 2616.17],
    [191., 2065.57, 3521.97],
    [192., 2476.22, 3893.92],
    [191., 1528.08, 5069.96],
    [192., 1891.42, 5069.96],
])

live_shifts = np.zeros((210, 2), dtype=np.float32)
frame_groups = defaultdict(list)
for t, y, x in fiducials:
    frame_groups[int(t)].append((y, x))

frame_ids = sorted(frame_groups.keys())
for i in range(0, len(frame_ids) - 1, 2):
    t_before = frame_ids[i]
    t_after  = frame_ids[i + 1]
    pts_before = np.mean(frame_groups[t_before], axis=0)
    pts_after  = np.mean(frame_groups[t_after],  axis=0)
    shift = pts_before - pts_after
    # Propagate shift from transition frame onwards
    for t in range(t_after, 210):
        live_shifts[t] += shift

# Combine live and fixed shifts
shifts = live_shifts.copy()
shifts[192:] += fixed_shifts[192:]
print('Shifts computed:', shifts.shape)

## 12 · Visualise shifts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(shifts[:, 1], label='dx', color='#978897')
ax.plot(shifts[:, 0], label='dy', color='#1a9641')
ax.axvline(48,  color='#d02c91', linestyle='--', label='Live-1 -> Live-2')
ax.axvline(192, color='#1a9641', linestyle='--', label='Live -> Fixed')
ax.legend()
ax.set(xlabel='Frame', ylabel='Shift (px)', title='Cumulative frame shifts')
sns.despine(ax=ax, offset=10)
plt.tight_layout()

## 13 · Apply shifts to images and segmentation

In [ ]:
images_shifted = da.stack([
    da.map_blocks(
        lambda img, dy=dy, dx=dx: np.stack(
            [ndi_shift(ch, shift=(dy, dx), order=1, mode='nearest') for ch in img], axis=0
        ),
        images[t], dtype=images.dtype
    )
    for t, (dy, dx) in enumerate(shifts)
])

seg_shifted_list = [
    ndi_shift(segmentation[t].astype(float), shift=(-dy, -dx), order=0, mode='nearest').astype(segmentation.dtype)
    for t, (dy, dx) in enumerate(tqdm(shifts))
]
seg_shifted = np.stack(seg_shifted_list)

viewer.add_labels(seg_shifted, name='Segmentation (shifted)')
viewer.add_image(np.stack([f.compute() for f in images_shifted]), channel_axis=1, name='Images (shifted)')

## 14 · Apply shifts to track coordinates

In [ ]:
def apply_shifts_to_tracks(df, shifts):
    """Subtract per-frame spatial shifts from track y/x coordinates."""
    df = df.copy()
    shift_arr = shifts[df['t'].values]
    df['y'] -= shift_arr[:, 0]
    df['x'] -= shift_arr[:, 1]
    return df

# Realign live_2 time indices to follow live_1
live_2_tracks['t'] += live_1_tracks['t'].max() + 1

aligned_live_1 = apply_shifts_to_tracks(live_1_tracks, shifts)
aligned_live_2 = apply_shifts_to_tracks(live_2_tracks, shifts)

all_aligned_tracks = pd.concat([aligned_live_1, aligned_live_2], ignore_index=True)

napari_tracks = all_aligned_tracks[['label_id', 't', 'y', 'x']].values
viewer.add_tracks(napari_tracks, name='Aligned tracks')

## TODO
- Reconcile fixed-alignment jump ~3 frames into the fixed sequence with the live-cell alignment.
- Validate track continuity across Live-1 / Live-2 / Fixed transitions.